# Autopilot: run the next most valuable job


**Open this one every session.** It works out what to run from what is already
in Drive, then runs it until the session budget is spent.

A free-tier lease is short and ends without warning, so the expensive mistake
is not the learning rate, it is spending a session on work already done or
leaving a lease idle while you decide. The plan is recomputed from Drive every
time, so it is correct after a preemption, after a manual run, and after a
session that got halfway through a job.

Order is cheapest-informative-first: S10 (one session, and it sets the batch
size for everything after it), then S2, then S3's **control arm before its DR
arm** (an orphaned baseline is still a usable no-DR number; an orphaned DR arm
answers no question at all), then the evaluation.

Every child process shares one XLA compilation cache on Drive, which is why
this is worth using even for a single job: measured on this codebase, a cold
compile of 6.45 s came back at 0.95 s from cache, and MJX's LEAP compile is
minutes rather than seconds.


---

### Before you run anything

1. **Runtime → Change runtime type → T4 GPU.** Every cell below assumes it.
2. **Keep this tab visible.** Free Colab disconnects an idle notebook after
   about 90 minutes and reclaims the runtime; `/content` does not survive it.
3. **The free tier has a quota you cannot see.** It is not published, it
   varies, and it is consumed by wall-clock GPU time whether or not you are
   computing. Expect a few hours a day, and expect to be cut off mid-run
   without warning. Every long-running cell here is written to survive that.

Checkpoints go to Google Drive, not to `/content`. That is the whole reason
the Drive cell exists, a run that checkpoints only to local disk loses
everything the moment the runtime is reclaimed, which on free Colab is the
normal way a session ends rather than an exceptional one.

## 1. Hardware, packages, Drive, code

In [ ]:
# --- what hardware did we actually get? ---
import subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout)

# A label is not hardware. A Kaggle session advertised as "T4 x2" reported a
# P100 to the driver, which has compute capability 6.0 and cannot run several
# things a T4 can. Record what the driver says and quote it as such; never
# write "measured on a T4" because the runtime menu said T4.

In [ ]:
# --- packages ---
# jax with CUDA is preinstalled on Colab GPU runtimes. mujoco-mjx is not, and
# installing it can drag in a CPU-only jax wheel that silently replaces the
# working one. So: install, then re-check the device, and only reinstall jax
# if the check fails.
import subprocess, sys

def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True)
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

sh(f'{sys.executable} -m pip install -q mujoco mujoco-mjx optax')

import importlib, jax
importlib.reload(jax)
if not any(d.platform == 'gpu' for d in jax.devices()):
    print('jax lost the GPU during install; reinstalling the CUDA wheel')
    sh(f'{sys.executable} -m pip install -q -U "jax[cuda12]"')
    raise SystemExit(
        'Reinstalled jax. Runtime -> Restart session, then run this cell '
        'again. (A restart is required: the CPU-only jax is already imported '
        'into this process and reimporting will not replace it.)')

In [ ]:
import jax, mujoco
print('jax     ', jax.__version__)
print('mujoco  ', mujoco.__version__)
print('devices ', jax.devices())
print('kind    ', getattr(jax.devices()[0], 'device_kind', '?'), '(as reported by the driver)')

# Hard stop, not a warning. On CPU a single mjx.step of the LEAP scene costs
# about 4 seconds and the compile runs past half an hour: a CPU session is not
# a slow run, it is no run.
assert any(d.platform == 'gpu' for d in jax.devices()), \
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then restart and re-run.'
print()
print('GPU OK')

In [ ]:
# --- Drive, for anything that must outlive this runtime ---
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/robotics-rl-portfolio')
DRIVE.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)

# One XLA compilation cache on Drive, shared by this process and every
# subprocess it starts. MJX compiles the LEAP scene in minutes and a
# preempted run re-pays that every session; cached, it is seconds. Set as
# environment variables rather than jax.config so child processes inherit it,
# and because the directory has to be known before the first compile.
os.environ['JAX_COMPILATION_CACHE_DIR'] = str(DRIVE / 'jax_cache')
os.environ['JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS'] = '0.5'
os.environ['JAX_COMPILATION_CACHE_MAX_SIZE'] = str(2_000_000_000)
(DRIVE / 'jax_cache').mkdir(parents=True, exist_ok=True)

_n = len([p for p in (DRIVE / 'jax_cache').rglob('*') if p.is_file()])
print('drive :', DRIVE)
print('local :', WORK)
print('cache :', _n, 'entries', '(empty: this session pays full compile cost)' if not _n else '')

In [ ]:
# --- code ---
import os, shutil, subprocess, sys
from pathlib import Path

SRC = WORK / 'robotics-rl-portfolio'
REPO = 'https://github.com/JacobEGarcia/robotics-rl-portfolio.git'

def sh(cmd, **kw):
    print('$', cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=True, **kw)

if SRC.exists():
    sh(f'cd {SRC} && git pull --ff-only')
else:
    sh(f'git clone --depth 1 {REPO} {SRC}')

need = SRC / 'colab/autopilot.py'
if not need.exists():
    # Fallback for code that is committed locally but not pushed. Tar the
    # repo on your machine, drop it in Drive, and this picks it up:
    #   tar czf portfolio.tgz --exclude=assets --exclude=runs .
    tgz = DRIVE / 'portfolio.tgz'
    if tgz.exists():
        print(f'{need} missing from the clone; unpacking {tgz} over it')
        sh(f'tar xzf {tgz} -C {SRC}')
    if not need.exists():
        raise SystemExit(
            f'{need} is not in the cloned repo and no {tgz} was found.\n'
            f'Push it from your machine:\n'
            f'    cd ~/Downloads/hermestes/robotics-rl-portfolio && git push origin main\n'
            f'or upload a tarball to {tgz}.')

os.environ['PYTHONPATH'] = str(SRC)
sys.path.insert(0, str(SRC))
print()
print('code OK at', SRC)

In [ ]:
# --- the robot models ---
# assets/menagerie is gitignored in the portfolio repo on purpose: Menagerie
# is 2.3 GB of third-party assets and is itself a git repo, so it is fetched
# rather than vendored. A sparse checkout gets what is needed in a few
# seconds instead of pulling all of it.
MENAGERIE = SRC / 'assets' / 'menagerie'
WANT = ['leap_hand', 'franka_emika_panda', 'unitree_z1',
        'unitree_go1', 'unitree_h1', 'shadow_hand', 'dynamixel_2r']

if not (MENAGERIE / 'leap_hand' / 'right_hand.xml').exists():
    shutil.rmtree(MENAGERIE, ignore_errors=True)
    MENAGERIE.parent.mkdir(parents=True, exist_ok=True)
    sh('git clone --depth 1 --filter=blob:none --sparse '
       'https://github.com/google-deepmind/mujoco_menagerie.git ' + str(MENAGERIE))
    sh(f'cd {MENAGERIE} && git sparse-checkout set ' + ' '.join(WANT))

missing = [w for w in WANT if not (MENAGERIE / w).exists()]
assert not missing, f'sparse checkout did not produce: {missing}'
print('models OK:', sorted(p.name for p in MENAGERIE.iterdir() if p.is_dir())[:12])

## 2. What should this session do?

Dry run first. Nothing is started, nothing is charged against the quota.

In [ ]:
import os, subprocess, sys
os.chdir(SRC)
subprocess.run([sys.executable, 'colab/autopilot.py',
                '--drive', str(DRIVE), '--plan'],
               env=dict(os.environ, PYTHONPATH=str(SRC)))

## 3. Run it

`SESSION_HOURS` is the budget for this whole notebook; `MAX_HOURS` is the
budget for any single training job, and should sit below where the free tier
tends to cut you off. A job that hits its budget writes a final checkpoint and
exits 0, so the next session resumes rather than restarts.

`NUM_ENVS` should be the knee that S10's throughput sweep measured, not the
largest batch that fits. Past the knee each extra environment buys under 10%
throughput while costing proportional memory and a longer compile, and on a
preemptible lease a longer compile is a direct loss.

Interrupting this cell is safe. The newest checkpoint is already in Drive.

In [ ]:
SESSION_HOURS = 3.0
MAX_HOURS     = 2.5
NUM_ENVS      = 2048     # use S10's measured knee

p = subprocess.Popen(
    [sys.executable, '-u', 'colab/autopilot.py',
     '--drive', str(DRIVE),
     '--session-hours', str(SESSION_HOURS),
     '--max-hours', str(MAX_HOURS),
     '--num-envs', str(NUM_ENVS)],
    cwd=str(SRC), env=dict(os.environ, PYTHONPATH=str(SRC)),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    p.terminate()
    print('\ninterrupted; the newest checkpoint is in Drive')
p.wait()

## 4. Where everything stands

In [ ]:
import json
from pathlib import Path

def latest(d):
    cks = sorted(Path(d).glob('ckpt_*.pkl')) if Path(d).exists() else []
    return int(cks[-1].stem.split('_')[1]) if cks else 0

print(f"{'job':<16}{'steps':>14}{'target':>14}")
for name, d, target in (
        ('S2',          DRIVE / 's2_checkpoints', 100_000_000),
        ('S3 baseline', DRIVE / 's3_checkpoints' / 'baseline', 30_000_000),
        ('S3 dr',       DRIVE / 's3_checkpoints' / 'dr', 30_000_000)):
    print(f'{name:<16}{latest(d):>14,}{target:>14,}')

s10 = DRIVE / 's10_results' / 'results'
print()
print('S10 results :', sorted(p.name for p in s10.glob('*.json')) if s10.exists() else 'none yet')
print('S3 eval     :', 'done' if (DRIVE / 's3_eval.json').exists() else 'pending')

cache = DRIVE / 'jax_cache'
n = len([p for p in cache.rglob('*') if p.is_file()]) if cache.exists() else 0
mb = sum(p.stat().st_size for p in cache.rglob('*') if p.is_file()) / 1e6 if n else 0
print(f'xla cache   : {n} entries, {mb:.0f} MB (saved compile time next session)')